# 00 · Funciones MTG (Scryfall)

**Este notebook solo define cosas; no descarga ni analiza nada.** Los otros dos lo cargan con:

```python
%run ./00_funciones.ipynb
```

| Sección | Qué trae |
|---|---|
| 1. Configuración | rutas, headers de Scryfall, dependencias |
| 2. API y bulk data | `api_get`, `actualizar_bulk`, `ultimo_parquet` |
| 3. Sets | `buscar_set`, `descargar_set`, `cargar_set` |
| 4. Visor de cartas | `mostrar_carta` |
| 5. Etiquetas para Limited | `preparar_cartas` (removal, trucos, mecánicas, evasión, fixing…) |
| 6. Análisis de set | tablas reutilizables: curva, dureza, removal, arquetipos, bombas, probabilidades |
| 7. Pool de sealed | `evaluar_pool` |
| 8. Reporte HTML | `reporte_set`, `mostrar_reporte` |

Regla de la casa: si una función se usa en más de un notebook, vive aquí.

In [ ]:
# ---------- 1. Configuración ----------
import importlib.util, subprocess, sys

# Instala solo lo que falte (DataLab ya trae pandas y requests)
_paquetes = {"duckdb": "duckdb", "zstandard": "zstandard", "PIL": "Pillow", "cairosvg": "cairosvg"}
_faltan = [pip for mod, pip in _paquetes.items() if importlib.util.find_spec(mod) is None]
if _faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_faltan])

import difflib, gzip, html, io, json, math, re, shutil, time
from datetime import date, datetime
from pathlib import Path

import duckdb
import pandas as pd
import requests
from IPython.display import HTML, display
from PIL import Image, ImageDraw, ImageFilter, ImageFont
try:  # cairosvg necesita libcairo del sistema; si no está, seguimos sin los íconos reales
    import cairosvg
except (ImportError, OSError):
    cairosvg = None

API = "https://api.scryfall.com"
# Scryfall exige User-Agent (nombre/versión de TU app) y Accept. Solo ASCII.
HEADERS = {
    "User-Agent": "KevinMtgLocal/0.2",
    "Accept": "application/json;q=0.9,*/*;q=0.8",
}
PAUSA_API = 0.1            # Scryfall pide 50-100 ms entre requests

DATA = Path("data")
RAW = DATA / "raw"          # archivo original ya descomprimido
PARQUET = DATA / "parquet"  # bulk en Parquet zstd
SETS = DATA / "sets"        # un Parquet por set (vía /cards/search)
REPORTES = Path("reportes")

esc = html.escape

## 2. API y bulk data

`actualizar_bulk()` hace todo el flujo: metadatos → descarga en streaming → descompresión (detecta gzip/zstd por los primeros bytes) → Parquet. Es idempotente: si el archivo de esa fecha ya existe, no hace nada.

In [ ]:
def api_get(ruta_o_url: str, params: dict | None = None, reintentos: int = 3) -> dict:
    """GET a Scryfall con pausa de cortesía, reintento en 429 y error legible."""
    url = ruta_o_url if ruta_o_url.startswith("http") else f"{API}{ruta_o_url}"
    for intento in range(1, reintentos + 1):
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        time.sleep(PAUSA_API)
        if r.status_code == 429 and intento < reintentos:
            time.sleep(2 * intento)
            continue
        if not r.ok:
            try:
                detalle = r.json().get("details")   # Scryfall explica el motivo aquí
            except ValueError:
                detalle = r.text[:300]
            raise requests.HTTPError(f"Scryfall {r.status_code} en {r.url}: {detalle}", response=r)
        return r.json()


def info_bulk(tipo: str) -> dict:
    """Metadatos del bulk pedido: oracle_cards, default_cards, all_cards..."""
    data = api_get("/bulk-data")["data"]
    for item in data:
        if item["type"] == tipo:
            return item
    raise ValueError(f"Tipo '{tipo}' no existe. Disponibles: {[i['type'] for i in data]}")


def elegir_uri(meta: dict) -> tuple[str, str]:
    """La API cambió nombres de campo con el tiempo: usa el primero disponible."""
    for clave in ("jsonl_download_uri", "download_uri", "json_download_uri"):
        if meta.get(clave):
            return clave, meta[clave]
    raise KeyError(f"No hay ningún campo de descarga en la respuesta: {sorted(meta)}")


def detectar_formato(path: Path) -> str:
    """Mira los primeros bytes: no confía en la extensión ni en los headers."""
    with open(path, "rb") as f:
        head = f.read(4)
    if head[:2] == b"\x1f\x8b":
        return "gzip"
    if head == b"\x28\xb5\x2f\xfd":
        return "zstd"
    return "plano"


def descomprimir(origen: Path, destino: Path) -> str:
    """Deja `destino` en texto plano y borra `origen`. Devuelve el formato detectado."""
    formato = detectar_formato(origen)
    if formato == "plano":
        origen.replace(destino)
        return formato
    with open(destino, "wb") as dst:
        if formato == "gzip":
            with gzip.open(origen, "rb") as src:
                shutil.copyfileobj(src, dst)
        else:
            import zstandard
            with open(origen, "rb") as src:
                zstandard.ZstdDecompressor().copy_stream(src, dst)
    origen.unlink()
    return formato


def descargar(url: str, destino: Path) -> None:
    """Descarga en streaming a un .part y luego descomprime."""
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_suffix(destino.suffix + ".part")
    with requests.get(url, headers=HEADERS, stream=True, timeout=60) as r:
        r.raise_for_status()
        bajado = 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
                bajado += len(chunk)
                print(f"\r  Descargado: {bajado / 1e6:,.1f} MB", end="")
        print()
    print(f"  Formato: {descomprimir(tmp, destino)} -> {destino.name}")


def a_parquet(json_path: Path, parquet_path: Path) -> int:
    """JSON o JSONL -> Parquet zstd. DuckDB infiere el esquema (incluye anidados).
    Devuelve el número de filas escritas."""
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    with duckdb.connect() as con:
        con.execute(f"""
            COPY (
                SELECT * FROM read_json_auto(
                    '{json_path.as_posix()}', sample_size = -1, maximum_object_size = 33554432)
            ) TO '{parquet_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """)
        return con.execute("SELECT count(*) FROM read_parquet(?)", [parquet_path.as_posix()]).fetchone()[0]


def limpiar_versiones(carpeta: Path, patron: str, conservar: int = 2) -> None:
    """Scryfall publica bulk a diario: borra versiones viejas para no llenar el workspace."""
    archivos = sorted(carpeta.glob(patron))
    for viejo in archivos[:-conservar] if conservar else []:
        viejo.unlink()
        print(f"  Borrado (versión vieja): {viejo}")


def actualizar_bulk(tipo: str = "oracle_cards", forzar: bool = False,
                    borrar_raw: bool = False, conservar: int = 2) -> Path:
    """Flujo completo: metadatos -> descarga -> Parquet. Devuelve la ruta del Parquet.
    - tipo: oracle_cards (1 fila por carta) | default_cards (todas las impresiones) | all_cards
    - borrar_raw: borra el JSONL tras convertir (ahorra espacio; el Parquet basta para analizar)"""
    meta = info_bulk(tipo)
    clave, uri = elegir_uri(meta)
    ext = ".jsonl" if "jsonl" in clave else ".json"
    fecha = meta["updated_at"][:10]
    json_path = RAW / f"{tipo}_{fecha}{ext}"
    parquet_path = PARQUET / f"{tipo}_{fecha}.parquet"

    tam = meta.get("compressed_size") or meta.get("size")
    print(f"{tipo} | actualizado {meta['updated_at']} | "
          f"{f'{tam / 1e6:.0f} MB comprimido' if tam else 'tamaño desconocido'} | campo {clave}")

    if parquet_path.exists() and not forzar:
        print(f"Parquet ya existe: {parquet_path}")
        return parquet_path
    if forzar or not json_path.exists():
        print("Descargando...")
        descargar(uri, json_path)
    print("Convirtiendo a Parquet...")
    n = a_parquet(json_path, parquet_path)
    print(f"OK: {n:,} filas -> {parquet_path}")

    if borrar_raw:
        json_path.unlink()
    limpiar_versiones(RAW, f"{tipo}_*", conservar)
    limpiar_versiones(PARQUET, f"{tipo}_*.parquet", conservar)
    return parquet_path


def ultimo_parquet(tipo: str = "oracle_cards") -> Path:
    """El Parquet más reciente de ese tipo (los nombres llevan fecha ISO, así que ordenan bien)."""
    archivos = sorted(PARQUET.glob(f"{tipo}_*.parquet"))
    if not archivos:
        raise FileNotFoundError(f"No hay {tipo}_*.parquet en {PARQUET}. Corre 01_descarga primero.")
    return archivos[-1]


def sql(consulta: str, **params) -> pd.DataFrame:
    """Atajo DuckDB -> pandas. Usa $nombre en la consulta y pásalo como keyword:
    sql("SELECT name FROM read_parquet($pq) WHERE cmc = $cmc", pq=ultimo_parquet(), cmc=3)"""
    params = {k: (v.as_posix() if isinstance(v, Path) else v) for k, v in params.items()}
    with duckdb.connect() as con:
        return con.execute(consulta, params).df()

## 3. Sets

Para analizar **un set** no hace falta el bulk: `/cards/search?q=e:<código>` trae las ~270 cartas en 2 páginas y queda cacheado en `data/sets/<código>.parquet`.

- `buscar_set("reality fracture")` o `buscar_set("fra")` → metadatos (código, fecha, tamaño).
- `cargar_set(...)` → DataFrame listo para análisis (ya pasado por `preparar_cartas`).

In [ ]:
def buscar_set(texto: str) -> dict:
    """Código exacto ('fra') o parte del nombre ('reality fracture') -> metadatos del set.
    Si hay varios (set principal, commander, tokens...) prioriza la expansión principal."""
    t = texto.strip().lower()
    sets = api_get("/sets")["data"]
    candidatos = [s for s in sets if s["code"] == t] or [s for s in sets if t in s["name"].lower()]
    if not candidatos:
        raise ValueError(f"No hay ningún set con código o nombre '{texto}'.")
    prioridad = {"expansion": 0, "core": 1, "draft_innovation": 2, "masters": 3}
    candidatos.sort(key=lambda s: (prioridad.get(s["set_type"], 9), bool(s.get("parent_set_code"))))
    elegido = candidatos[0]
    if len(candidatos) > 1:
        otros = ", ".join(f"{s['code']} ({s['set_type']})" for s in candidatos[1:6])
        print(f"Uso {elegido['code']} · otros que coinciden: {otros}")
    return elegido


def descargar_set(texto: str, forzar: bool = False, consulta_extra: str = "") -> Path:
    """Baja todas las cartas del set (una por carta, sin variantes de arte) a Parquet.
    consulta_extra: sintaxis Scryfall adicional, ej. 'is:booster'."""
    meta = buscar_set(texto)
    code = meta["code"]
    SETS.mkdir(parents=True, exist_ok=True)
    parquet_path, jsonl_path = SETS / f"{code}.parquet", SETS / f"{code}.jsonl"
    (SETS / f"{code}_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
    if parquet_path.exists() and not forzar:
        print(f"Set ya descargado: {parquet_path} (forzar=True para refrescar precios)")
        return parquet_path

    params = {"q": f"e:{code} {consulta_extra}".strip(), "unique": "cards",
              "order": "set", "include_extras": "false"}
    cartas, pagina = [], api_get("/cards/search", params)
    while True:
        cartas += pagina["data"]
        print(f"\r  {meta['name']}: {len(cartas)}/{pagina.get('total_cards', '?')} cartas", end="")
        if not pagina.get("has_more"):
            break
        pagina = api_get(pagina["next_page"])
    print()
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for c in cartas:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    n = a_parquet(jsonl_path, parquet_path)
    jsonl_path.unlink()
    print(f"OK: {n} cartas -> {parquet_path}")
    return parquet_path


def meta_set(code: str) -> dict:
    p = SETS / f"{code}_meta.json"
    return json.loads(p.read_text()) if p.exists() else {"code": code, "name": code.upper()}


def cargar_set(texto: str, forzar: bool = False) -> tuple[pd.DataFrame, dict]:
    """(DataFrame preparado, metadatos). Usa el Parquet local si existe y no hay internet."""
    t = texto.strip().lower()
    code = t
    for m in SETS.glob("*_meta.json"):              # permite buscar por nombre sin internet
        info = json.loads(m.read_text())
        if t in (info.get("code", ""), info.get("name", "").lower()) or t in info.get("name", "").lower():
            code = info["code"]
            break
    local = SETS / f"{code}.parquet"
    if not (local.exists() and not forzar):
        local = descargar_set(texto, forzar=forzar)
        code = local.stem
    crudo = sql("SELECT * FROM read_parquet($pq)", pq=local)
    return preparar_cartas(crudo), meta_set(code)

## 4. Visor de cartas

`mostrar_carta("Muldrotha")` busca en el último `oracle_cards`. Para buscar en un set descargado: `mostrar_carta("nombre", fuente=SETS / "fra.parquet")`. Acepta listas.

In [ ]:
SYM = "https://svgs.scryfall.io/card-symbols/{}.svg"
FORMATOS = ["commander", "brawl", "standard", "pioneer", "modern", "legacy", "vintage", "pauper", "oathbreaker"]
ESTADO = {"legal": ("Legal", "ok"), "not_legal": ("No legal", "no"),
          "banned": ("Baneada", "ban"), "restricted": ("Restringida", "res")}

CSS_CARTA = """
<style>
.mtg{font-family:-apple-system,'Segoe UI',Roboto,sans-serif;background:#14171c;color:#e8eaed;border-radius:14px;padding:22px;
     display:flex;gap:26px;flex-wrap:wrap;max-width:880px;box-shadow:0 4px 18px rgba(0,0,0,.35);margin-bottom:14px}
.mtg .imgs{display:flex;flex-direction:column;gap:12px;flex:0 0 300px;max-width:100%}
.mtg .carta{width:300px;max-width:100%;border-radius:4.5%;box-shadow:0 6px 18px rgba(0,0,0,.55)}
.mtg .info{flex:1 1 320px;min-width:280px}
.mtg .fila{display:flex;justify-content:space-between;align-items:center;gap:10px}
.mtg h2{margin:0;font-size:26px;color:#f3d77a;font-weight:700}
.mtg h3{margin:0;font-size:18px;color:#f3d77a}
.mtg .sym{height:1.05em;width:1.05em;vertical-align:-0.15em;margin:0 1px}
.mtg .fila .sym{height:1.3em;width:1.3em}
.mtg .bloque{background:#1d2229;border:1px solid #2b323c;border-radius:10px;padding:12px 14px;margin-top:12px}
.mtg .tipo{color:#aab2bd;font-size:14px;margin-bottom:8px;font-style:italic}
.mtg .texto{font-size:15px;line-height:1.5}
.mtg .stats,.mtg .grid{display:flex;flex-wrap:wrap;gap:10px;margin-top:10px}
.mtg .grid{margin-top:14px}
.mtg .stat{background:#232a33;border-radius:8px;padding:8px 12px;min-width:90px}
.mtg .stat span{display:block;font-size:11px;text-transform:uppercase;letter-spacing:.06em;color:#8a94a1}
.mtg .stat b{font-size:16px;font-weight:600}
.mtg .chip{display:inline-block;background:#2b3340;border-radius:999px;padding:3px 11px;font-size:12px;margin:10px 6px 0 0}
.mtg .kw{color:#f3d77a;border:1px solid #4a4326;background:#26241a}
.mtg .titulo{margin:18px 0 8px;font-size:11px;text-transform:uppercase;letter-spacing:.08em;color:#8a94a1}
.mtg .legs{display:grid;grid-template-columns:repeat(auto-fill,minmax(112px,1fr));gap:8px}
.mtg .leg{border-radius:8px;padding:7px 10px;font-size:12px}
.mtg .leg span{display:block;color:#c5ccd6}.mtg .leg b{font-size:13px}
.mtg .leg.ok{background:#173324;border:1px solid #2a6b45}.mtg .leg.ok b{color:#5fd68c}
.mtg .leg.no{background:#22262c;border:1px solid #333a44}.mtg .leg.no b{color:#8a94a1}
.mtg .leg.ban{background:#3a1b1f;border:1px solid #7a2f38}.mtg .leg.ban b{color:#ff7b86}
.mtg .leg.res{background:#3a3016;border:1px solid #7a6420}.mtg .leg.res b{color:#f0c24b}
.mtg .pie{margin-top:16px;font-size:12px;color:#8a94a1}.mtg .pie a{color:#7db4ff;text-decoration:none}
</style>"""


def _simbolos(t: str) -> str:
    """{2}{B}{G}, {T}, {W/P}... -> íconos oficiales de Scryfall (con fallback a texto)."""
    return re.sub(
        r"\{([^}]+)\}",
        lambda m: (f'<img class="sym" src="{SYM.format(m.group(1).replace("/", ""))}" '
                   f'alt="{m.group(0)}" onerror="this.outerHTML=this.alt">'),
        t)


def _reglas(t) -> str:
    t = esc(t or "")
    t = re.sub(r"\(([^)]*)\)", r"<i>(\1)</i>", t)          # texto recordatorio en cursiva
    return _simbolos(t).replace("\n", "<br>")


def _stat(etiqueta, valor) -> str:
    return (f'<div class="stat"><span>{etiqueta}</span><b>{valor}</b></div>'
            if valor not in (None, "") else "")


def _bloque(x: dict, con_nombre: bool) -> str:
    cab = (f'<div class="fila"><h3>{esc(x.get("name", ""))}</h3>'
           f'<span>{_simbolos(x.get("mana_cost") or "")}</span></div>') if con_nombre else ""
    pt = f'{x["power"]} / {x["toughness"]}' if x.get("power") is not None else None
    stats = "".join([_stat("Fuerza / Resistencia", pt),
                     _stat("Lealtad", x.get("loyalty")),
                     _stat("Defensa", x.get("defense"))])
    return (f'<div class="bloque">{cab}<div class="tipo">{esc(x.get("type_line") or "")}</div>'
            f'<div class="texto">{_reglas(x.get("oracle_text"))}</div>'
            f'{"<div class=stats>" + stats + "</div>" if stats else ""}</div>')


def _buscar_carta(nombre: str, fuente: Path) -> tuple[dict, list[str]]:
    with duckdb.connect() as con:
        def consultar(q, params):
            cur = con.execute(q, params)
            cols = [d[0] for d in cur.description]
            return [dict(zip(cols, fila)) for fila in cur.fetchall()]
        pq = fuente.as_posix()
        exactas = consultar(
            "SELECT * FROM read_parquet(?) WHERE lower(name) = lower(?) "
            "OR lower(name) LIKE lower(?) || ' // %' LIMIT 1", [pq, nombre, nombre])
        parciales = [] if exactas else consultar(
            "SELECT * FROM read_parquet(?) WHERE name ILIKE '%' || ? || '%' "
            "ORDER BY edhrec_rank NULLS LAST LIMIT 6", [pq, nombre])
    if not (exactas or parciales):
        raise ValueError(f"No encontré ninguna carta parecida a '{nombre}' en {fuente.name}.")
    return (exactas or parciales)[0], [x["name"] for x in parciales[1:]]


def _render_carta(c: dict, otras: list[str]) -> str:
    caras = c.get("card_faces") or []
    if c.get("image_uris"):
        imgs = [c["image_uris"].get("normal")]
    else:                                                    # doble cara: una imagen por cara
        imgs = [(f.get("image_uris") or {}).get("normal") for f in caras]
    imgs_html = "".join(f'<img class="carta" src="{u}" alt="{esc(c["name"])}">' for u in imgs if u)

    bloques = "".join(_bloque(f, True) for f in caras) if len(caras) > 1 else _bloque(c, False)
    costo = _simbolos(c.get("mana_cost") or "") if len(caras) <= 1 else ""

    ci = c.get("color_identity") or []
    identidad = "".join(_simbolos("{" + k + "}") for k in ci) or _simbolos("{C}") + " Incolora"
    precio = (c.get("prices") or {}).get("usd")
    generales = "".join([
        _stat("Valor de maná", int(c["cmc"]) if c.get("cmc") is not None else None),
        _stat("Identidad de color", identidad),
        _stat("Rango EDHREC", f'#{c["edhrec_rank"]:,}' if c.get("edhrec_rank") else None),
        _stat("Precio (USD)", f"${precio}" if precio else None),
        _stat("Rareza", (c.get("rarity") or "").capitalize()),
    ])
    keywords = "".join(f'<span class="chip kw">{esc(k)}</span>' for k in (c.get("keywords") or []))

    leg = c.get("legalities") or {}
    legalidades = "".join(
        f'<div class="leg {ESTADO.get(leg.get(f), ("—", "no"))[1]}"><span>{f.capitalize()}</span>'
        f'<b>{ESTADO.get(leg.get(f), ("—", "no"))[0]}</b></div>'
        for f in FORMATOS if f in leg)

    pie = " · ".join(filter(None, [
        f'Ilustración: {esc(c["artist"])}' if c.get("artist") else "",
        f'{esc(c["set_name"])} ({esc(c["set"]).upper()})' if c.get("set_name") else "",
        f'<a href="{esc(c["scryfall_uri"])}" target="_blank">Ver en Scryfall</a>' if c.get("scryfall_uri") else "",
    ]))
    otras_html = f'<div class="pie">También coincide: {esc(", ".join(otras))}</div>' if otras else ""

    return f"""{CSS_CARTA}
<div class="mtg">
  <div class="imgs">{imgs_html}</div>
  <div class="info">
    <div class="fila"><h2>{esc(c["name"])}</h2><span>{costo}</span></div>
    {bloques}
    <div class="grid">{generales}</div>
    <div>{keywords}</div>
    <div class="titulo">Legalidad</div>
    <div class="legs">{legalidades}</div>
    <div class="pie">{pie}<br>Datos e imágenes: Scryfall. Magic: The Gathering © Wizards of the Coast.</div>
    {otras_html}
  </div>
</div>"""


def mostrar_carta(nombre, fuente: Path | None = None) -> None:
    """Imagen + datos de una carta (nombre exacto o parcial, en inglés). Acepta lista de nombres."""
    if isinstance(nombre, (list, tuple)):
        for n in nombre:
            mostrar_carta(n, fuente)
        return
    carta, otras = _buscar_carta(nombre, Path(fuente) if fuente else ultimo_parquet())
    display(HTML(_render_carta(carta, otras)))

## 5. Etiquetas para Limited

`preparar_cartas(df)` aplana cartas de doble cara y agrega columnas útiles para Limited. Las etiquetas salen de **regex sobre el texto Oracle** (sin texto recordatorio), así que son una clasificación automática: sirve para filtrar rápido, no es un rating.

| Columna | Qué significa |
|---|---|
| `color` | `W U B R G` = monocolor, `M` = multicolor, `C` = incolora |
| `par` | par de colores de las doradas (`WU`, `UB`…) |
| `interaccion` | `removal` (destroy/exile/-X/-X/aura que anula), `daño`, `pelea`, `rebote/tap`, `contra`, `masivo` o vacío |
| `daño_max` | cuánto daño o -X/-X hace el removal (para cruzarlo con resistencias) |
| `truco` | instantáneo o flash que potencia/protege a una criatura |
| `evasion`, `robo`, `fixing` | booleanos |
| `mecanicas` | lista de mecánicas detectadas (`MECANICAS`, editable) |
| `puntaje_bomba` | heurística para ordenar raras/míticas (ver `puntuar_bomba`) |

In [ ]:
ORDEN_COLOR = "WUBRGMC"
NOMBRE_COLOR = {"W": "Blanco", "U": "Azul", "B": "Negro", "R": "Rojo", "G": "Verde",
                "M": "Multicolor", "C": "Incolora"}
RAREZAS = ["common", "uncommon", "rare", "mythic"]
RAREZA_ES = {"common": "Común", "uncommon": "Infrecuente", "rare": "Rara", "mythic": "Mítica"}

# Mecánicas a rastrear: nombre -> regex sobre texto en minúsculas (sin recordatorio).
# Las de Reality Fracture salen de las guías de prerelease; agrega o quita libremente.
MECANICAS = {
    "Surveil": r"\bsurveil",
    "Scry": r"\bscry\b",
    "Threshold (7+ en cementerio)": r"\bthreshold\b|seven or more cards in your graveyard",
    "Flashback": r"\bflashback\b",
    "Prepare": r"\bprepared?\b",
    "Empower Jace": r"\bempower jace\b",
    "Heartwood": r"\bheartwood\b",
    "Cadet": r"\bcadets?\b",
    "Landcycling": r"landcycling",
    "Ganar vidas": r"\bgain \w+ life|you gain life|\blifelink\b",
    "Contadores +1/+1": r"\+1/\+1 counter",
    "Hechizo no-criatura": r"noncreature spell|instant or sorcery spell",
    "Muere / sacrificio": r"\bdies\b|\bsacrifice\b",
    "Tokens": r"\bcreate\b[^.]*\btoken",
}

_N = r"(\d+|x)"
_REMOVAL = [  # (etiqueta, regex) en orden de prioridad
    ("masivo", r"(destroy|exile) all (other )?creatures|damage to each creature|all creatures get -|each creature gets -"),
    ("removal", r"\b(destroy|exile) (up to (one|two) )?(another )?target (?:[a-z,\- ]*?)(creature|planeswalker|permanent)(?! cards?)"),
    ("removal", rf"target creature[^.]*gets -{_N}/-([1-9]\d*|x)"),
    ("removal", r"base power and toughness 0/0"),
    ("removal", r"(target|each) opponent sacrifices an? (creature|nonland permanent|attacking)"),
    ("removal", r"enchanted creature (can't attack or block|loses all abilities|gets -)"),
    ("removal", r"target (other )?nonland permanent[^.]*(top or bottom|bottom) of (their|its owner's) library"
                r"|owner of (up to one )?(other )?target nonland permanent puts it on"),
    ("daño", rf"deals {_N} damage to (any target|(up to one )?(another )?target [a-z ]*?(creature|planeswalker))"),
    ("pelea", r"\bfights? (another |up to one )?target|damage equal to its power to (that|another target|up to one target|target)"),
    ("contra", r"counter target [a-z ]*?spell"),
    ("rebote/tap", r"return (up to one )?target [a-z ]*?(creature|permanent)[^.]*owner'?s'? hand"
                   r"|tap target creature[^.]*(doesn't|don't) untap|doesn't untap during its controller's next untap"
                   r"|base power and toughness 1/1|choose target nonland permanent\. its owner may put it on top"),
]
_TRUCO = (r"target creature( you control)?[^.]*gets \+\d+/\+\d+|creatures you control get \+"
          r"|gains? (indestructible|hexproof|protection|first strike|double strike|deathtouch|lifelink)")
_EVASION = {"flying", "menace", "trample", "skulk", "shadow", "fear", "intimidate", "horsemanship"}


def _lista(x) -> list:
    """DuckDB puede devolver listas, arrays numpy o None/NaN: normaliza a list."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return []
    return list(x) if not isinstance(x, (str, dict)) else [x]


def _num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def _sin_recordatorio(t: str) -> str:
    return re.sub(r"\([^)]*\)", "", t or "")


def etiquetar(texto: str, tipo: str, keywords: list[str]) -> dict:
    t = _sin_recordatorio(texto).lower()
    kw = {k.lower() for k in keywords}
    inter, dmg = "", None
    for etiqueta, patron in _REMOVAL:
        m = next((m for m in re.finditer(patron, t) if "noncreature" not in m.group(0)), None)
        if m:
            inter = etiqueta
            dano = re.search(rf"deals {_N} damage", m.group(0))
            menos = re.search(rf"gets -{_N}/-{_N}", m.group(0))
            valor = dano.group(1) if dano else (menos.group(2) if menos else None)
            if valor is not None:
                dmg = "X" if valor == "x" else int(valor)
            elif etiqueta in ("removal", "masivo"):
                dmg = "∞"          # destroy/exile/sacrificio: mata sin importar la resistencia
            break
    es_instantaneo = "instant" in tipo.lower() or "flash" in kw
    return {
        "interaccion": inter,
        "daño_max": dmg,
        "truco": bool(es_instantaneo and not inter and re.search(_TRUCO, t)),
        "evasion": bool(kw & _EVASION) or "can't be blocked" in t,
        "robo": bool(re.search(r"\bdraws? (a|an|two|three|x|that many) cards?", t)),
        "mecanicas": [n for n, p in MECANICAS.items() if re.search(p, t)],
    }


_KW_PROTECCION = ("ward", "hexproof", "indestructible")   # difícil de remover con removal normal
_KW_COMBATE = ("deathtouch", "double strike", "lifelink")  # relevantes en combate y no evasión


def _lineas_propias(texto: str) -> list[str]:
    """Separa el oracle text en líneas/segmentos (por salto de línea y coma) en minúscula.
    Sirve para detectar palabras clave que son la propia habilidad de la carta, no un efecto
    que la carta le concede a otra cosa (ej. "target creature gains ward" no empieza con "ward",
    pero "Ward—Sacrifice three permanents." sí)."""
    return [seg.strip().lower() for linea in texto.split("\n") for seg in linea.split(",")]


def puntuar_bomba(r: dict) -> float:
    """Heurística simple para ORDENAR raras/míticas; no reemplaza leer la carta."""
    s = 0.0
    tipo = r["tipo"]
    s += 4 if "Planeswalker" in tipo else 0
    s += {"masivo": 4, "removal": 3, "daño": 2.5, "pelea": 2, "contra": 1, "rebote/tap": 1}.get(r["interaccion"], 0)
    s += 1 if r["robo"] else 0
    s += 1 if r["evasion"] else 0
    if r["fuerza"] is not None and r["cmc"]:
        s += max(0, min(2, (r["fuerza"] + (r["resistencia"] or 0)) / (2 * r["cmc"]) - 1) * 2)
    # Una criatura enorme cierra la partida aunque su "tasa" por maná no sea eficiente
    # (la fórmula de arriba penaliza el top end; esto lo compensa un poco).
    if r["fuerza"] is not None and r["fuerza"] >= 6:
        s += min(2, (r["fuerza"] - 5) * 0.3)
    t = r["texto"].lower()
    s += 1 if re.search(r"\b(whenever|at the beginning of)\b", t) else 0
    s += 0.5 if "when you cast this spell" in t else 0
    s += 1 if re.search(r"\bcreate\b[^.]*creature token", t) else 0
    segs = _lineas_propias(r["texto"])
    s += 1 if any(seg.startswith(p) for seg in segs for p in _KW_PROTECCION) else 0
    s += 0.5 if any(seg.startswith(p) for seg in segs for p in _KW_COMBATE) else 0
    return round(s, 1)


def preparar_cartas(crudo: pd.DataFrame) -> pd.DataFrame:
    """Scryfall crudo -> una fila por carta con columnas planas + etiquetas de Limited."""
    filas = []
    for c in crudo.to_dict("records"):
        caras = [f for f in _lista(c.get("card_faces")) if isinstance(f, dict)]
        cara0 = caras[0] if caras else {}
        def campo(k):
            v = c.get(k)
            return v if v is not None and not (isinstance(v, float) and math.isnan(v)) else cara0.get(k)
        texto = "\n//\n".join(f.get("oracle_text") or "" for f in caras) if caras else (c.get("oracle_text") or "")
        colores = _lista(c.get("colors")) or sorted({x for f in caras for x in _lista(f.get("colors"))})
        colores = sorted(colores, key="WUBRG".index)
        tipo = campo("type_line") or ""
        imgs = c.get("image_uris") if isinstance(c.get("image_uris"), dict) else (cara0.get("image_uris") or {})
        keywords = _lista(c.get("keywords"))
        r = {
            "nombre": c["name"],
            "costo": campo("mana_cost") or "",
            "cmc": int(c.get("cmc") or 0),
            "tipo": tipo,
            "tipo_base": next((b for b in ["Creature", "Planeswalker", "Instant", "Sorcery", "Battle",
                                            "Enchantment", "Artifact", "Land"] if b in tipo.split("//")[0]), "Otro"),
            "rareza": c.get("rarity"),
            "colores": "".join(colores),
            "color": colores[0] if len(colores) == 1 else ("M" if colores else "C"),
            "par": "".join(colores) if len(colores) == 2 else "",
            "fuerza": _num(campo("power")),
            "resistencia": _num(campo("toughness")),
            "texto": texto,
            "keywords": keywords,
            "numero": c.get("collector_number"),
            "booster": bool(c.get("booster")) if c.get("booster") is not None else True,
            "fixing": len([m for m in _lista(c.get("produced_mana")) if m in "WUBRG"]) >= 2
                      or bool(re.search(r"mana of any color|search your library for a basic land|landcycling|heartwood",
                                        texto.lower())),
            "imagen": (imgs or {}).get("normal"),
            "url": c.get("scryfall_uri"),
            "precio_usd": _num((c.get("prices") or {}).get("usd")) if isinstance(c.get("prices"), dict) else None,
        }
        r.update(etiquetar(texto, tipo, keywords))
        r["puntaje_bomba"] = puntuar_bomba(r)
        filas.append(r)
    df = pd.DataFrame(filas)
    if not df["booster"].any():
        # Pasa con sets recién salidos: Scryfall aún no marca qué cartas van en sobres.
        print("ℹ Scryfall no marca ninguna carta como 'booster' todavía: asumo que todas salen en sobres.")
        df["booster"] = True
    df["rareza"] = pd.Categorical(df["rareza"], RAREZAS, ordered=True)
    df["color"] = pd.Categorical(df["color"], list(ORDEN_COLOR), ordered=True)
    return df.sort_values(["color", "rareza", "cmc", "nombre"]).reset_index(drop=True)

## 6. Análisis de set (tablas reutilizables)

Cada función devuelve un `DataFrame`, así que puedes explorarlas en el notebook de análisis y también las usa el reporte HTML. Por defecto miran **comunes + infrecuentes que salen en sobres**, que son el 80% de un pool de sealed.

`ARQUETIPOS` resume los 10 pares según la [guía oficial de prerelease](https://magic.wizards.com/en/news/feature/reality-fracture-prerelease-guide); las cartas señal (doradas infrecuentes) se detectan solas desde los datos.

In [ ]:
ARQUETIPOS = {  # par -> (nombre, tema). Fuente: guía oficial de prerelease de Reality Fracture.
    "WU": ("Azorius", "Surveil/scry y aggro-tempo"),
    "UB": ("Dimir", "Cementerio: 7+ cartas (threshold)"),
    "BR": ("Rakdos", "Daño directo a rivales y criaturas"),
    "RG": ("Gruul", "Rampa con tokens Heartwood"),
    "WG": ("Selesnya", "Ganar vidas → contadores +1/+1"),
    "WB": ("Orzhov", "Desgaste: criaturas pequeñas, premio cuando mueren"),
    "UR": ("Izzet", "Prowess: gatillos de hechizos no-criatura"),
    "BG": ("Golgari", "Criaturas que generan valor y recursión"),
    "WR": ("Boros", "Ejército: contadores +1/+1 y tokens"),
    "UG": ("Simic", "Empower Jace: valor del token planeswalker"),
}
PARES = list(ARQUETIPOS)

# Aproximación de un Play Booster (14 cartas). Edita si tienes el desglose oficial del set.
# Ranuras esperadas por rareza, sumando las 2 wildcards (una foil) con un reparto aproximado.
PLAY_BOOSTER = {"common": 7.0 + 1.1, "uncommon": 3.0 + 0.6, "rare": 6 / 7 + 0.2, "mythic": 1 / 7 + 0.06}


def _limited(df: pd.DataFrame, rarezas=("common", "uncommon")) -> pd.DataFrame:
    d = df[df["booster"] & ~df["tipo"].str.contains("Basic Land")]
    return d[d["rareza"].isin(rarezas)] if rarezas else d


def composicion(df: pd.DataFrame) -> pd.DataFrame:
    t = pd.crosstab(df["color"], df["rareza"], margins=True, margins_name="Total")
    t.index = [NOMBRE_COLOR.get(i, i) for i in t.index]
    return t.rename(columns=RAREZA_ES)


def perfil_colores(df: pd.DataFrame, rarezas=("common", "uncommon")) -> pd.DataFrame:
    """Profundidad de cada color en C/U: qué tanto te da si terminas en él."""
    d = _limited(df, rarezas)
    es_criat = d["tipo_base"].eq("Creature")
    d = pd.DataFrame({
        "color": d["color"],
        "es_criat": es_criat,
        "removal_duro": d["interaccion"].isin(["removal", "masivo"]),
        "dano_pelea": d["interaccion"].isin(["daño", "pelea"]),
        "otra": d["interaccion"].isin(["contra", "rebote/tap"]),
        "truco": d["truco"].astype(bool),
        "evasivas": d["evasion"].astype(bool) & es_criat,
        "robo": d["robo"].astype(bool),
        "fixing": d["fixing"].astype(bool),
        "cmc_criat": d["cmc"].astype(float).where(es_criat),
    })
    t = (d.groupby("color", observed=True)
          .agg(cartas=("es_criat", "size"), criaturas=("es_criat", "sum"), removal_duro=("removal_duro", "sum"),
               **{"daño/pelea": ("dano_pelea", "sum"), "otra_interaccion": ("otra", "sum")},
               trucos=("truco", "sum"), evasivas=("evasivas", "sum"), robo=("robo", "sum"),
               fixing=("fixing", "sum"), cmc_medio_criat=("cmc_criat", "mean"))
          .round({"cmc_medio_criat": 2}))
    t.index = [NOMBRE_COLOR.get(i, i) for i in t.index]
    return t


def curva_criaturas(df: pd.DataFrame, rarezas=("common", "uncommon")) -> pd.DataFrame:
    d = _limited(df, rarezas)
    d = d[d["tipo_base"] == "Creature"].assign(cmc_b=lambda x: x["cmc"].clip(upper=6).astype(int).astype(str).replace("6", "6+"))
    t = pd.crosstab(d["color"], d["cmc_b"])
    t.index = [NOMBRE_COLOR.get(i, i) for i in t.index]
    return t


def tamano_por_cmc(df: pd.DataFrame, rarezas=("common", "uncommon")) -> pd.DataFrame:
    """Cuerpo típico por coste: te dice qué es 'grande' en este formato."""
    d = _limited(df, rarezas)
    d = d[(d["tipo_base"] == "Creature") & d["fuerza"].notna()]
    d = d.assign(cmc_b=d["cmc"].clip(upper=6))
    return (d.groupby("cmc_b").agg(criaturas=("nombre", "size"), fuerza_media=("fuerza", "mean"),
                                   resistencia_media=("resistencia", "mean"),
                                   pct_evasivas=("evasion", "mean"))
            .round(2).assign(pct_evasivas=lambda x: (x["pct_evasivas"] * 100).round(0))
            .rename_axis("cmc (6 = 6+)"))


def dureza(df: pd.DataFrame, rarezas=("common", "uncommon")) -> pd.DataFrame:
    """% de criaturas C/U que mueren a N de daño (resistencia <= N)."""
    d = _limited(df, rarezas)
    r = d.loc[(d["tipo_base"] == "Creature") & d["resistencia"].notna(), "resistencia"]
    return pd.DataFrame({"daño": range(1, 7),
                         "pct_criaturas_que_mueren": [round((r <= n).mean() * 100, 1) for n in range(1, 7)]})


def tabla_interaccion(df: pd.DataFrame, rarezas=("common", "uncommon", "rare", "mythic")) -> pd.DataFrame:
    d = df[df["booster"] & (df["interaccion"] != "") & df["rareza"].isin(rarezas)]
    cu = _limited(df)
    res = cu.loc[cu["tipo_base"] == "Creature", "resistencia"].dropna()
    mata = d["daño_max"].map(lambda x: 100.0 if x == "∞" else
                             round((res <= x).mean() * 100) if isinstance(x, (int, float)) else None)
    return (d.assign(instantaneo=d["tipo"].str.contains("Instant") | d["keywords"].map(lambda k: "Flash" in k),
                     pct_criaturas_CU_que_mata=mata)
            [["color", "rareza", "nombre", "costo", "cmc", "interaccion", "daño_max", "instantaneo",
              "pct_criaturas_CU_que_mata"]]
            .sort_values(["color", "rareza", "cmc"]))


def hoja_trucos(df: pd.DataFrame, rarezas=("common", "uncommon")) -> pd.DataFrame:
    """Todo lo que el rival puede hacer con maná abierto (instantáneos y flash) en C/U."""
    d = _limited(df, rarezas)
    d = d[d["tipo"].str.contains("Instant") | d["keywords"].map(lambda k: "Flash" in k)]
    rol = [i or ("truco" if t else ("criatura con flash" if tb == "Creature" else "otro"))
           for i, t, tb in zip(d["interaccion"], d["truco"], d["tipo_base"])]
    corto = d["texto"].map(lambda t: re.sub(r"\s+", " ", _sin_recordatorio(t)).strip()[:160])
    return (d.assign(rol=rol, texto_corto=corto)[["color", "cmc", "costo", "nombre", "rareza", "rol", "texto_corto"]]
            .sort_values(["color", "cmc"]))


def mecanicas_por_color(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df["booster"]].explode("mecanicas").dropna(subset=["mecanicas"]).reset_index(drop=True)
    t = pd.crosstab(d["mecanicas"], d["color"])
    t.columns = [NOMBRE_COLOR.get(c, c) for c in t.columns]
    t["Total"] = t.sum(axis=1)
    return t.sort_values("Total", ascending=False)


def keywords_top(df: pd.DataFrame, n: int = 25) -> pd.DataFrame:
    """Keywords que Scryfall reconoce: aquí aparecen mecánicas que no estén en MECANICAS."""
    k = df[df["booster"]].explode("keywords")["keywords"].dropna().value_counts().head(n)
    return k.rename_axis("keyword").reset_index(name="cartas")


def arquetipos(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df["booster"]]
    cu = _limited(df)
    filas = []
    for par, (nombre, tema) in ARQUETIPOS.items():
        oro = d[d["par"].map(lambda p: set(p) == set(par))]
        en_colores = cu[cu["colores"].map(lambda c: bool(c) and set(c) <= set(par))]
        filas.append({
            "par": par, "gremio": nombre, "tema": tema,
            "señales (doradas infrecuentes)": ", ".join(oro.loc[oro["rareza"] == "uncommon", "nombre"]),
            "doradas": len(oro),
            "raras/míticas doradas": ", ".join(oro.loc[oro["rareza"].isin(["rare", "mythic"]), "nombre"]),
            "C/U jugables en el par": len(en_colores),
            "removal C/U en el par": int(en_colores["interaccion"].isin(["removal", "masivo", "daño", "pelea"]).sum()),
        })
    return pd.DataFrame(filas)


def bombas(df: pd.DataFrame, n: int = 24) -> pd.DataFrame:
    d = df[df["booster"] & df["rareza"].isin(["rare", "mythic"]) & (df["tipo_base"] != "Land")]
    return (d.sort_values("puntaje_bomba", ascending=False).head(n)
            [["nombre", "color", "rareza", "costo", "tipo", "interaccion", "puntaje_bomba"]])


def probabilidades(df: pd.DataFrame, sobres: int = 6) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(copias esperadas de UNA carta concreta por rareza, removal esperado en tu pool por color).
    Usa PLAY_BOOSTER, que es aproximado: tómalo como orden de magnitud."""
    d = df[df["booster"] & ~df["tipo"].str.contains("Basic Land")]   # las básicas van en su propia ranura
    n = d["rareza"].value_counts()
    lam = {r: sobres * PLAY_BOOSTER[r] / n[r] for r in RAREZAS if n.get(r)}
    t1 = pd.DataFrame({
        "cartas distintas": [n.get(r, 0) for r in lam],
        "copias esperadas de una carta concreta": [round(v, 3) for v in lam.values()],
        "prob. de abrir ≥1": [f"{(1 - math.exp(-v)) * 100:.1f}%" for v in lam.values()],
    }, index=[RAREZA_ES[r] for r in lam])
    rem = d[d["interaccion"].isin(["removal", "masivo", "daño", "pelea"]) & d["rareza"].isin(lam.keys())]
    t2 = (rem.assign(esperado=rem["rareza"].map(lambda r: lam[r]).astype(float))
          .groupby("color", observed=True)["esperado"].sum().round(2)
          .rename("removal esperado en tu pool").to_frame())
    t2.index = [NOMBRE_COLOR.get(i, i) for i in t2.index]
    return t1, t2

## 7. Pool de sealed

Pega tu pool (una carta por línea, con o sin cantidad: `2 Last Gasp` o `Last Gasp`) y `evaluar_pool` te dice qué pares de colores tienen más material. El puntaje por carta es simple y transparente (`peso_carta`): úsalo para decidir entre 2-3 opciones, no como verdad absoluta.

In [ ]:
def leer_pool(texto: str, df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Texto pegado -> (filas del set repetidas por cantidad, líneas que no se reconocieron)."""
    indice = {}
    for i, n in enumerate(df["nombre"]):
        indice[n.lower()] = i
        for parte in n.split(" // "):
            indice.setdefault(parte.lower(), i)
    filas, sin_match, aproximados = [], [], []
    for linea in texto.strip().splitlines():
        linea = re.sub(r"\s*\([A-Z0-9]{2,5}\)\s*\d*\s*$", "", linea.strip())   # quita '(FRA) 123' de Arena/MTGO
        if not linea or linea.lower() in ("deck", "sideboard", "commander"):
            continue
        m = re.match(r"^(\d+)\s*x?\s+(.+)$", linea, flags=re.I)
        cant, nombre = (int(m.group(1)), m.group(2)) if m else (1, linea)
        clave = nombre.lower().strip()
        if clave not in indice:
            parecido = difflib.get_close_matches(clave, indice.keys(), n=1, cutoff=0.88)
            if not parecido:
                sin_match.append(linea)
                continue
            aproximados.append(f"{nombre} → {df['nombre'].iloc[indice[parecido[0]]]}")
            clave = parecido[0]
        filas += [indice[clave]] * cant
    if aproximados:
        print("Revisa estas coincidencias aproximadas:\n  " + "\n  ".join(aproximados))
    return df.iloc[filas].reset_index(drop=True), sin_match


def peso_carta(r) -> float:
    """1 por ser jugable + premios por lo que más gana partidas de sealed."""
    w = 1.0
    w += {"removal": 2.5, "masivo": 2.5, "daño": 2, "pelea": 1.5, "contra": 0.75, "rebote/tap": 0.75}.get(r["interaccion"], 0)
    w += 0.5 if r["evasion"] and r["tipo_base"] == "Creature" else 0
    w += 0.5 if r["robo"] else 0
    w += min(r["puntaje_bomba"], 6) / 3 if r["rareza"] in ("rare", "mythic") else 0
    return round(w, 2)


def _pips(costo: str, color: str) -> int:
    return len(re.findall(r"\{[^}]*" + color + r"[^}]*\}", costo or ""))


def evaluar_pool(df: pd.DataFrame, texto: str) -> dict:
    pool, sin_match = leer_pool(texto, df)
    if sin_match:
        print(f"⚠ No reconocí {len(sin_match)} líneas: {sin_match[:10]}")
    if pool.empty:
        return {"pool": pool, "pares": pd.DataFrame(), "splash": pd.DataFrame(), "fixing": [], "sin_match": sin_match}
    pool = pool.assign(peso=pool.apply(peso_carta, axis=1))
    jugables = pool[pool["tipo_base"] != "Land"]
    filas = []
    for par in PARES:
        en_par = jugables[jugables["colores"].map(lambda c: set(c) <= set(par))]   # incluye incoloras
        mejores = en_par.sort_values("peso", ascending=False).head(23)
        criat = mejores[mejores["tipo_base"] == "Creature"]
        filas.append({
            "par": par, "gremio": ARQUETIPOS[par][0],
            "puntaje (top 23)": round(mejores["peso"].sum(), 1),
            "jugables": len(en_par), "criaturas (top 23)": len(criat),
            "removal": int(en_par["interaccion"].isin(["removal", "masivo", "daño", "pelea"]).sum()),
            "raras/míticas": int(en_par["rareza"].isin(["rare", "mythic"]).sum()),
            "curva criaturas 1-2/3/4/5+": "/".join(str(x) for x in [
                (criat["cmc"] <= 2).sum(), (criat["cmc"] == 3).sum(), (criat["cmc"] == 4).sum(), (criat["cmc"] >= 5).sum()]),
            "faltan para 23": max(0, 23 - len(en_par)),
        })
    pares = pd.DataFrame(filas).sort_values("puntaje (top 23)", ascending=False).reset_index(drop=True)

    # Splash: cartas fuertes con 1 solo símbolo del tercer color, para los 3 mejores pares
    splash = []
    for par in pares["par"].head(3):
        fuera = jugables[jugables["colores"].map(lambda c: bool(c) and not set(c) <= set(par) and len(set(c) - set(par)) == 1)]
        for _, r in fuera.iterrows():
            extra = (set(r["colores"]) - set(par)).pop()
            if _pips(r["costo"], extra) == 1 and r["peso"] >= 2.5:
                splash.append({"para": par, "splash": extra, "nombre": r["nombre"], "costo": r["costo"], "peso": r["peso"]})
    fixing = pool[pool["fixing"]]["nombre"].tolist()
    splash = pd.DataFrame(splash).drop_duplicates() if splash else pd.DataFrame()
    fixing = sorted(set(fixing))
    return {"pool": pool, "pares": pares, "splash": splash, "fixing": fixing, "sin_match": sin_match}

## 8. Reporte web

El reporte es un **sitio estático**: el front-end vive en la carpeta `web/` (HTML + CSS + JS, sin dependencias) y Python solo genera los **datos** en JSON.

| Función | Salida | Para qué |
|---|---|---|
| `exportar_sitio(df, meta, pool)` | `docs/` (index.html, assets/, data/<set>.json) | GitHub Pages. Varios sets conviven; `?set=fra` elige uno |
| `reporte_set(df, meta, pool)` | `reportes/limited_<set>.html` (un solo archivo) | Abrir con doble clic o verlo aquí con `mostrar_reporte` |
| `datos_reporte(df, meta, pool)` | `dict` | Por si quieres usar los datos en otro lado |

Requiere la carpeta `web/` en el workspace (junto a los notebooks).

In [ ]:
# ---------- 8.1 Rutas del front-end ----------
WEB = Path("web")      # plantilla del sitio: index.html + assets/app.css + assets/app.js
SITIO = Path("docs")   # salida para GitHub Pages (Settings → Pages → rama main, carpeta /docs)


def _vacio(v) -> bool:
    return v is None or (isinstance(v, float) and math.isnan(v)) or v == ""


def _limpio(v):
    """Valores JSON-seguros: NaN -> None, numpy -> python."""
    if v is None:
        return None
    if hasattr(v, "item") and not isinstance(v, (list, dict, str)):
        v = v.item()
    if isinstance(v, float):
        return None if math.isnan(v) else round(v, 2)
    return v


def _filas(df: pd.DataFrame) -> list[dict]:
    return [{k: _limpio(v) for k, v in r.items()} for r in df.to_dict("records")]


# ---------- 8.2 Textos calculados ----------
def _insights(df, perfil, dur, tam, arq) -> list[dict]:
    """Conclusiones en lenguaje natural calculadas desde los datos (se recalculan con cada set).
    "icono" es un ícono genérico fijo por tipo de insight (no depende del color); los colores
    relevantes se incrustan como símbolos de maná {W}/{U}/... dentro del propio texto."""
    inv = {v: k for k, v in NOMBRE_COLOR.items()}
    mono = perfil.loc[[NOMBRE_COLOR[c] for c in "WUBRG" if NOMBRE_COLOR[c] in perfil.index]]
    rem = (mono["removal_duro"] + mono["daño/pelea"]).sort_values(ascending=False)
    ev = mono["evasivas"].sort_values(ascending=False)
    robo = mono["robo"].sort_values(ascending=False)
    p = dict(zip(dur["daño"], dur["pct_criaturas_que_mueren"]))
    out = [
        ("bolt", "Dónde está el removal",
         f"{{{inv[rem.index[0]]}}} {rem.index[0]} ({rem.iloc[0]}) y {{{inv[rem.index[1]]}}} {rem.index[1]} ({rem.iloc[1]}) concentran el removal común e infrecuente; "
         f"{rem.index[-1]} es el más pobre ({rem.iloc[-1]}). Abrir removal de esos colores es buena razón para jugarlos."),
        ("heartcrack", "Qué tan frágiles son las criaturas",
         f"{p.get(2, 0):.0f}% muere a 2 de daño, {p.get(3, 0):.0f}% a 3 y {p.get(4, 0):.0f}% a 4. "
         f"Resistencia 4+ te saca del alcance de casi todo el daño barato."),
        ("wind", "Evasión",
         f"{{{inv[ev.index[0]]}}} {ev.index[0]} tiene más criaturas evasivas (vuelo, amenaza, arrollar o imbloqueables: {ev.iloc[0]}); "
         f"{{{inv[ev.index[-1]]}}} {ev.index[-1]} la que menos ({ev.iloc[-1]}). Guarda bloqueadores con alcance o vuelo contra {ev.index[0].lower()}."),
        ("cards", "Ventaja de cartas",
         f"{{{inv[robo.index[0]]}}} {robo.index[0]} es el color que más roba ({robo.iloc[0]} cartas C/U). En un formato lento eso gana partidas largas."),
    ]
    for c in (2, 3):
        if c in tam.index:
            out.append(("ruler", f"Criatura típica de coste {c}",
                        f"{tam.loc[c, 'fuerza_media']:.1f}/{tam.loc[c, 'resistencia_media']:.1f} de media "
                        f"({int(tam.loc[c, 'criaturas'])} criaturas, {tam.loc[c, 'pct_evasivas']:.0f}% con evasión). "
                        f"Algo por encima de esto ya es un cuerpo bueno."))
    top = arq.sort_values("removal C/U en el par", ascending=False).iloc[0]
    out.append(("duo", "Par con más removal",
                f"{{{top['par'][0]}}}{{{top['par'][1]}}} {top['gremio']} reúne {top['removal C/U en el par']} cartas de removal comunes e infrecuentes."))
    fix = int(df[df["booster"] & df["fixing"]].shape[0])
    out.append(("flask", "Mana fixing",
                f"{fix} cartas arreglan maná (tierras duales comunes, landcycling, tokens Heartwood para rojo-verde). "
                f"{'Un splash de 1–2 cartas fuertes es razonable.' if fix >= 12 else 'Splashear es arriesgado.'}"))
    return [{"icono": i, "titulo": t, "texto": x} for i, t, x in out]


GLOSARIO = {  # basado en el texto recordatorio oficial de las cartas del set
    "Empower Jace": ("U", "Pon N contadores de lealtad en tu token Jace. Si no tienes uno, primero creas un planeswalker "
                          "Jace azul con «−1: Surveil 1» y «−3: Roba una carta». Cuantas más cartas lo alimentan, más valor saca."),
    "Prepare": ("U", "Criaturas con un hechizo pegado (carta «Criatura // Hechizo»). Mientras la criatura esté «preparada» "
                     "puedes lanzar una copia del hechizo, y eso la deja sin preparar. Es una carta que vale por dos."),
    "Heartwood": ("G", "Token artefacto rojo y verde con «{T}: Añade {R} o {G}». Rampa y arreglo, pero solo para rojo-verde; "
                       "también cuenta como artefacto para cartas que los sacrifican."),
    "Surveil": ("B", "Mira las N cartas de arriba y pon cualquiera en el cementerio. Filtra robos y llena el cementerio para Threshold."),
    "Threshold": ("B", "Se activa con 7+ cartas en tu cementerio. En 40 cartas cuesta llegar sin surveil o mill: no lo fuerces."),
    "Flashback": ("R", "Puedes volver a lanzar la carta desde el cementerio pagando su coste de flashback; luego se exilia."),
    "Cadet": ("W", "Token de criatura incolora 2/2 Wizard Soldier. Muchas cartas los crean y cuentan como Wizards."),
    "Behold": ("U", "Coste adicional: muestra un Jace de tu mano o elige uno que controles (a veces con la alternativa de pagar maná)."),
    "Landcycling": ("C", "Paga y descarta la carta para buscar una tierra básica. Es hechizo tarde y tierra si te falta maná."),
}

# Mecánicas nuevas de este set (no existían antes en Magic); el resto del glosario
# son mecánicas ya conocidas que Reality Fracture reutiliza.
NOVEDADES = {
    "Empower Jace": "acumula lealtad hasta crear tu planeswalker Jace",
    "Prepare": "criaturas con un hechizo instantáneo pegado, listo para lanzar",
    "Heartwood": "token artefacto que produce maná rojo o verde",
    "Behold": "coste adicional: mostrar o controlar un Jace",
    "Cadet": "token 2/2 Wizard Soldier que crean varias cartas",
}


# ---------- 8.3 Datos del reporte (JSON que consume web/assets/app.js) ----------
def datos_reporte(df: pd.DataFrame, meta: dict, pool: dict | None = None,
                  fuentes: list[tuple[str, str]] | None = None) -> dict:
    code = meta.get("code", "set")
    b = df[df["booster"]]
    cu = _limited(df)
    inv = {v: k for k, v in NOMBRE_COLOR.items()}
    perfil, tam, dur, arq = perfil_colores(df), tamano_por_cmc(df), dureza(df), arquetipos(df)
    res_cu = cu.loc[cu["tipo_base"] == "Creature", "resistencia"].dropna()

    cartas = []
    for r in b.to_dict("records"):
        dmg = r["daño_max"]
        if dmg == "∞":
            mata = 100
        elif isinstance(dmg, (int, float)) and not _vacio(dmg):
            mata = round(float((res_cu <= dmg).mean()) * 100)
        else:
            mata = None
        cartas.append({
            "n": r["nombre"], "c": r["costo"], "cmc": int(r["cmc"]), "t": r["tipo"], "tb": r["tipo_base"],
            "x": r["texto"], "xs": re.sub(r"\s+", " ", _sin_recordatorio(r["texto"])).strip()[:170],
            "k": str(r["color"]), "par": r["par"], "r": str(r["rareza"]),
            "pt": (f'{r["fuerza"]:g}/{r["resistencia"]:g}' if not _vacio(r["fuerza"]) and not _vacio(r["resistencia"]) else ""),
            "img": r["imagen"] if isinstance(r["imagen"], str) else None, "u": r["url"],
            "inter": r["interaccion"], "dmg": _limpio(dmg), "mata": mata, "truco": bool(r["truco"]),
            "ev": bool(r["evasion"]), "robo": bool(r["robo"]), "fix": bool(r["fixing"]), "mec": list(r["mecanicas"]),
            "instant": "Instant" in r["tipo"].split("//")[0] or "Flash" in list(r["keywords"]),
            # hechizo instantáneo de una carta Prepare
            "prep": "//" in r["tipo"] and "Instant" in r["tipo"].split("//")[-1],
            "score": _limpio(r["puntaje_bomba"]), "pu": _limpio(r["precio_usd"]),
        })

    perf = perfil.rename_axis("n").reset_index()
    perf = (perf.assign(color=perf["n"].map(inv))
                .rename(columns={"daño/pelea": "dano_pelea", "otra_interaccion": "otra", "cmc_medio_criat": "cmc_medio"})
                .drop(columns="n"))
    comp = pd.crosstab(b["color"].astype(str), b["rareza"].astype(str)).reindex(columns=RAREZAS, fill_value=0)
    comp = comp.reindex([k for k in ORDEN_COLOR if k in comp.index])
    comp_rows = [{"color": k, **{r: int(v) for r, v in fila.items()}, "total": int(fila.sum())} for k, fila in comp.iterrows()]
    comp_rows.append({"color": "Total", **{r: int(comp[r].sum()) for r in RAREZAS}, "total": int(comp.to_numpy().sum()), "_total": True})
    curva = curva_criaturas(df)
    curva_rows = [{"color": inv.get(n, "C"), **{str(c): int(v) for c, v in fila.items()}} for n, fila in curva.iterrows()]
    mec = b.explode("mecanicas").dropna(subset=["mecanicas"]).reset_index(drop=True)
    mec_t = pd.crosstab(mec["mecanicas"], mec["color"].astype(str))
    mec_rows = sorted(({"mecanica": m, **{k: int(v) for k, v in fila.items()}, "total": int(fila.sum())}
                       for m, fila in mec_t.iterrows()), key=lambda x: -x["total"])
    tot_mec = {m["mecanica"]: m["total"] for m in mec_rows}
    prob1, prob2 = probabilidades(df)
    inv_r = {v: k for k, v in RAREZA_ES.items()}

    datos = {
        "version": 2,
        "generado": datetime.now().strftime("%d/%m/%Y"),
        "set": {"code": code, "name": meta.get("name", code.upper()), "released_at": meta.get("released_at"),
                "icon": meta.get("icon_svg_uri") or f"https://svgs.scryfall.io/sets/{code}.svg"},
        "kpis": [
            {"label": "Cartas del set", "valor": len(b),
             "nota": f"{(b['rareza'] == 'common').sum()} comunes · {(b['rareza'] == 'uncommon').sum()} infrecuentes"},
            {"label": "Removal C/U", "valor": int(cu["interaccion"].isin(["removal", "masivo", "daño", "pelea"]).sum()),
             "nota": "destruir, exiliar, daño o pelea"},
            {"label": "Mueren a 2 de daño", "valor": f"{dur.loc[dur['daño'] == 2, 'pct_criaturas_que_mueren'].iloc[0]:.0f}%",
             "nota": "de las criaturas C/U"},
            {"label": "Mueren a 3 de daño", "valor": f"{dur.loc[dur['daño'] == 3, 'pct_criaturas_que_mueren'].iloc[0]:.0f}%",
             "nota": "de las criaturas C/U"},
            {"label": "Trucos combate C/U", "valor": int(cu["truco"].sum()), "nota": "instantáneos de combate"},
            {"label": "Mana fixing", "valor": int(b["fixing"].sum()), "nota": "duales, landcycling, Heartwood"},
        ],
        "cards": cartas,
        "tablas": {
            "perfil": _filas(perf),
            "composicion": comp_rows,
            "dureza": [{"dano": int(d), "pct": float(p)} for d, p in zip(dur["daño"], dur["pct_criaturas_que_mueren"])],
            "tamano": _filas(tam.reset_index().set_axis(["coste", "criaturas", "fuerza", "resistencia", "pct_evasivas"], axis=1)),
            "curva": curva_rows,
            "mecanicas": mec_rows,
            "keywords": _filas(keywords_top(df, 40)),
            "arquetipos": [{"par": a["par"], "gremio": a["gremio"], "tema": a["tema"], "doradas": int(a["doradas"]),
                            "disponibles": int(a["C/U jugables en el par"]), "removal": int(a["removal C/U en el par"])}
                           for _, a in arq.iterrows()],
            "probabilidades": {
                "sobres": 6,
                "rareza": [{"r": inv_r[i], "distintas": int(f["cartas distintas"]), "prob": f["prob. de abrir ≥1"]}
                           for i, f in prob1.iterrows()],
                "removal": [{"color": inv.get(i, "C"), "esperado": float(v)}
                            for i, v in prob2["removal esperado en tu pool"].items()],
            },
        },
        "insights": _insights(df, perfil, dur, tam, arq),
        "glosario": [{"nombre": n, "color": c, "texto": t,
                      "cartas": next((v for k, v in tot_mec.items() if k.lower().startswith(n.split()[0].lower())), None)}
                     for n, (c, t) in GLOSARIO.items()],
        "novedades": [{"nombre": n, "texto": t} for n, t in NOVEDADES.items()],
        "reutilizadas": [n for n in GLOSARIO if n not in NOVEDADES],
        "fuentes": [{"t": t, "u": u} for t, u in (fuentes or [])],
        "pool": None,
    }

    if pool and not pool["pool"].empty:
        ids = {c["n"]: i for i, c in enumerate(cartas)}
        p = pool["pool"]
        cant = p["nombre"].value_counts()
        datos["pool"] = {
            "pares": _filas(pool["pares"].rename(columns={
                "puntaje (top 23)": "puntaje", "criaturas (top 23)": "criaturas", "raras/míticas": "raras",
                "curva criaturas 1-2/3/4/5+": "curva", "faltan para 23": "faltan"})),
            "splash": _filas(pool["splash"]) if not pool["splash"].empty else [],
            "fixing": list(pool["fixing"]),
            "cartas": [{"id": ids[r["nombre"]], "cant": int(cant[r["nombre"]]), "peso": float(r["peso"])}
                       for r in p.drop_duplicates("nombre").to_dict("records") if r["nombre"] in ids],
        }
    return datos


def _json(datos: dict) -> str:
    return json.dumps(datos, ensure_ascii=False, separators=(",", ":"))


def _plantilla(web: Path) -> Path:
    web = Path(web)
    if not (web / "index.html").exists():
        raise FileNotFoundError(f"No encuentro la plantilla en {web.resolve()}. "
                                "Sube la carpeta web/ del repo al workspace (junto a los notebooks).")
    return web


COLORES_ORDEN = ["W", "U", "B", "R", "G"]
_PIP_ESTILO = {
    "W": {"bg": (248, 246, 229), "fg": (42, 36, 18)},
    "U": {"bg": (47, 123, 196), "fg": (234, 244, 255)},
    "B": {"bg": (43, 36, 48), "fg": (217, 205, 255)},
    "R": {"bg": (211, 69, 58), "fg": (255, 240, 234)},
    "G": {"bg": (61, 138, 82), "fg": (237, 255, 242)},
}


def _fuente(tam: int, negrita: bool = True) -> ImageFont.FreeTypeFont:
    """Fuente serif para la imagen de Open Graph; si el entorno no trae ninguna
    de las candidatas usa la bitmap por defecto de Pillow (siempre disponible)."""
    candidatos = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if negrita else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf" if negrita else "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
        "C:\\Windows\\Fonts\\georgiab.ttf" if negrita else "C:\\Windows\\Fonts\\georgia.ttf",
        "/Library/Fonts/Georgia Bold.ttf" if negrita else "/Library/Fonts/Georgia.ttf",
    ]
    for c in candidatos:
        if Path(c).exists():
            return ImageFont.truetype(c, tam)
    try:
        return ImageFont.load_default(size=tam)
    except TypeError:
        return ImageFont.load_default()


def _svg_a_imagen(url: str, tam: int) -> "Image.Image | None":
    """Descarga y rasteriza un SVG de Scryfall (ícono del set o símbolo de maná)
    a una imagen cuadrada de tam×tam px con fondo transparente. Devuelve None si
    falla algo (sin red, sin cairosvg, SVG raro): quien llama cae a un dibujo
    propio en vez de reventar la exportación del sitio."""
    if cairosvg is None:
        return None
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        png = cairosvg.svg2png(bytestring=r.content, output_width=tam, output_height=tam)
        return Image.open(io.BytesIO(png)).convert("RGBA")
    except Exception:
        return None


def _og_imagen(meta: dict, tamano: int = 1200) -> Image.Image:
    """Imagen cuadrada 1:1 para las tarjetas de Open Graph/Twitter (lo que se ve
    al pegar el link): el emblema del set en el centro y los 5 símbolos de color
    alrededor, conectados en pentágono. No depende de red: se dibuja con Pillow,
    no usa los SVG reales de Scryfall (para eso habría que rasterizarlos)."""
    BG_TOP, BG_BOTTOM, GLOW = (42, 31, 77), (10, 7, 20), (155, 120, 255)
    LINE_SPOKE, LINE_EDGE = (244, 241, 255, 140), (201, 190, 250, 107)
    EMBLEM_FILL, EMBLEM_BORDER = (42, 32, 82), (201, 190, 250, 140)
    CRACK_A, CRACK_B = (217, 205, 255, 170), (185, 166, 255, 150)
    TITLE_COLOR, SUBTITLE_COLOR = (238, 233, 255), (185, 166, 255)
    PIP_BORDER, PIP_RING = (21, 15, 46), (201, 190, 250, 90)

    ss = 2  # supersample: líneas, círculos y texto salen suaves al reducir al final
    s = tamano * ss
    cx, cy = s // 2, round(s * (500 / 1200))
    radio, pip_r, emb_r = round(s * (320 / 1200)), round(s * (60 / 1200)), round(s * (120 / 1200))

    base = Image.new("RGB", (s, s), BG_BOTTOM)
    bd = ImageDraw.Draw(base)
    for y in range(s):
        t = y / s
        bd.line([(0, y), (s, y)], fill=tuple(round(BG_TOP[i] + (BG_BOTTOM[i] - BG_TOP[i]) * t) for i in range(3)))

    glow = Image.new("RGBA", (s, s), (0, 0, 0, 0))
    gr = round(s * 0.42)
    ImageDraw.Draw(glow).ellipse([cx - gr, -gr, cx + gr, gr], fill=(*GLOW, 80))
    glow = glow.filter(ImageFilter.GaussianBlur(s // 12))
    base = Image.alpha_composite(base.convert("RGBA"), glow)

    lines = Image.new("RGBA", (s, s), (0, 0, 0, 0))
    ld = ImageDraw.Draw(lines)
    angulos = [-90, -18, 54, 126, 198]
    pts = [(cx + radio * math.cos(math.radians(a)), cy + radio * math.sin(math.radians(a))) for a in angulos]
    for p in pts:
        ld.line([(cx, cy), p], fill=LINE_SPOKE, width=max(2, 3 * ss))
    for i in range(5):
        ld.line([pts[i], pts[(i + 1) % 5]], fill=LINE_EDGE, width=max(2, 3 * ss))
    base = Image.alpha_composite(base, lines)

    # emblema central: círculo con "grietas" y el código del set
    ed = emb_r * 2
    icono_set = _svg_a_imagen(
        meta.get("icon_svg_uri") or f"https://svgs.scryfall.io/sets/{meta.get('code', 'set')}.svg", ed)
    emb = Image.new("RGBA", (ed, ed), (0, 0, 0, 0))
    ed_draw = ImageDraw.Draw(emb)
    ed_draw.ellipse([0, 0, ed, ed], fill=(*EMBLEM_FILL, 255), outline=EMBLEM_BORDER, width=max(2, 4 * ss))
    if icono_set is not None:
        relleno = round(ed * 0.22)
        emb.alpha_composite(icono_set.resize((ed - relleno * 2, ed - relleno * 2), Image.LANCZOS), (relleno, relleno))
    else:
        for ang, col in ((28, CRACK_A), (-42, CRACK_B), (-8, CRACK_A)):
            largo = ed * 1.3
            dx, dy = largo * math.cos(math.radians(ang)) / 2, largo * math.sin(math.radians(ang)) / 2
            ed_draw.line([(ed / 2 - dx, ed / 2 - dy), (ed / 2 + dx, ed / 2 + dy)], fill=col, width=max(2, 4 * ss))
    mascara = Image.new("L", (ed, ed), 0)
    ImageDraw.Draw(mascara).ellipse([0, 0, ed, ed], fill=255)
    emb.putalpha(Image.composite(emb.split()[3], Image.new("L", (ed, ed), 0), mascara))
    if icono_set is None:
        codigo = str(meta.get("code", "SET")).upper()
        ed_draw.text((ed / 2, ed / 2), codigo, font=_fuente(round(s * (22 / 1200))), fill=(244, 241, 255, 255),
                     anchor="mm", stroke_width=max(1, ss), stroke_fill=(15, 10, 30, 200))
    base.alpha_composite(emb, (cx - emb_r, cy - emb_r))

    for (px, py), c in zip(pts, COLORES_ORDEN):
        pd_size = pip_r * 2 + 20 * ss
        pd = Image.new("RGBA", (pd_size, pd_size), (0, 0, 0, 0))
        pdd, cc = ImageDraw.Draw(pd), pd_size / 2
        pdd.ellipse([cc - pip_r - 10 * ss] * 2 + [cc + pip_r + 10 * ss] * 2, fill=PIP_RING)
        simbolo = _svg_a_imagen(f"https://svgs.scryfall.io/card-symbols/{c}.svg", pip_r * 2)
        if simbolo is not None:
            pd.alpha_composite(simbolo, (round(cc - pip_r), round(cc - pip_r)))
        else:
            est = _PIP_ESTILO[c]
            pdd.ellipse([cc - pip_r, cc - pip_r, cc + pip_r, cc + pip_r], fill=(*est["bg"], 255),
                        outline=PIP_BORDER, width=max(2, 4 * ss))
            pdd.text((cc, cc), c, font=_fuente(round(s * (44 / 1200))), fill=(*est["fg"], 255), anchor="mm")
        base.alpha_composite(pd, (round(px - cc), round(py - cc)))

    out = ImageDraw.Draw(base)
    ty = round(s * (960 / 1200))
    out.text((cx, ty), str(meta.get("name", meta.get("code", "Set"))), font=_fuente(round(s * (100 / 1200))),
             fill=TITLE_COLOR, anchor="mm")
    out.text((cx, ty + round(s * (95 / 1200))), "ANALYSIS", font=_fuente(round(s * (32 / 1200)), negrita=False),
             fill=SUBTITLE_COLOR, anchor="mm")

    return base.convert("RGB").resize((tamano, tamano), Image.LANCZOS)


def _og_datos(df: pd.DataFrame, meta: dict, url_sitio: str = "", imagen_url: str | None = None) -> dict:
    """Título, descripción, ícono e imagen para el favicon y las tarjetas de
    Open Graph / Twitter Card (lo que se ve al pegar el link en Slack, Discord, etc.).
    `imagen_url` es la imagen 1:1 generada por `_og_imagen` (ver `exportar_sitio`); sin
    ella cae a la carta con más puntaje de bomba, y si tampoco hay, al ícono del set."""
    name = meta.get("name", meta.get("code", "Set").upper())
    icon = meta.get("icon_svg_uri") or f"https://svgs.scryfall.io/sets/{meta.get('code', 'set')}.svg"
    if imagen_url:
        imagen = imagen_url
    else:
        b = df[df["booster"]]
        bombas = b[b["rareza"].isin(["rare", "mythic"]) & (b["tipo_base"] != "Land")].sort_values("puntaje_bomba", ascending=False)
        imagen = next((r["imagen"] for _, r in bombas.iterrows() if isinstance(r["imagen"], str)), None) or icon
    return {
        "title": f"{name} · Guía de Limited",
        "desc": f"Guía de Limited y prerelease para {name}: removal, arquetipos, curva de maná "
                f"y las mejores cartas del set, calculada desde datos de Scryfall.",
        "image": imagen,
        "icon": icon,
        "url": url_sitio,
    }


def _con_og(doc: str, og: dict) -> str:
    return (doc.replace("%%OG_TITLE%%", esc(og["title"], quote=True))
               .replace("%%OG_DESC%%", esc(og["desc"], quote=True))
               .replace("%%OG_IMAGE%%", esc(og["image"], quote=True))
               .replace("%%OG_ICON%%", esc(og["icon"], quote=True))
               .replace("%%OG_URL%%", esc(og["url"], quote=True)))


# ---------- 8.4 Salidas ----------
def exportar_sitio(df: pd.DataFrame, meta: dict, pool: dict | None = None, fuentes=None,
                   destino: Path = SITIO, web: Path = WEB, por_defecto: bool = True,
                   url_sitio: str = "https://kgavec.github.io/MTG_reality_fracture") -> Path:
    """Sitio estático para GitHub Pages: destino/index.html + assets/ + data/<set>.json.
    Varios sets conviven: data/sets.json lista los disponibles y ?set=<código> elige uno."""
    web, destino = _plantilla(web), Path(destino)
    (destino / "data").mkdir(parents=True, exist_ok=True)
    shutil.copytree(web / "assets", destino / "assets", dirs_exist_ok=True)
    datos = datos_reporte(df, meta, pool, fuentes)
    code = datos["set"]["code"]
    _og_imagen(meta).save(destino / "assets" / f"og-{code}.png")
    og = _og_datos(df, meta, url_sitio, imagen_url=f"{url_sitio}/assets/og-{code}.png")
    doc = _con_og((web / "index.html").read_text(encoding="utf-8"), og)
    (destino / "index.html").write_text(doc, encoding="utf-8")
    (destino / ".nojekyll").write_text("")        # GitHub Pages: servir tal cual, sin Jekyll
    (destino / "data" / f"{code}.json").write_text(_json(datos), encoding="utf-8")

    indice_path = destino / "data" / "sets.json"
    indice = json.loads(indice_path.read_text()) if indice_path.exists() else {"default": code, "sets": []}
    indice["sets"] = [s for s in indice["sets"] if s["code"] != code] + [
        {"code": code, "name": datos["set"]["name"], "released_at": datos["set"]["released_at"]}]
    if por_defecto:
        indice["default"] = code
    indice_path.write_text(json.dumps(indice, ensure_ascii=False, indent=1))
    peso = sum(f.stat().st_size for f in destino.rglob("*") if f.is_file()) / 1e3
    print(f"Sitio listo en {destino}/ ({peso:.0f} KB). Prueba local: python -m http.server -d {destino}")
    return destino / "index.html"


def reporte_set(df: pd.DataFrame, meta: dict, pool: dict | None = None, ruta: Path | None = None,
                fuentes=None, web: Path = WEB) -> Path:
    """Versión de UN archivo (CSS, JS y datos incrustados): se abre con doble clic o dentro del notebook."""
    web = _plantilla(web)
    doc = (web / "index.html").read_text(encoding="utf-8")
    css = (web / "assets" / "app.css").read_text(encoding="utf-8")
    js = (web / "assets" / "app.js").read_text(encoding="utf-8")
    doc = _con_og(doc, _og_datos(df, meta, url_sitio=""))  # sin URL pública: es un archivo local
    datos = _json(datos_reporte(df, meta, pool, fuentes)).replace("</", "<\\/")
    doc = (doc.replace('<link rel="stylesheet" href="assets/app.css">', f"<style>{css}</style>")
              .replace('<script src="assets/app.js" defer></script>', f"<script>{js}</script>")
              .replace("<!--DATA-->", f"<script>window.__DATA__={datos}</script>"))
    REPORTES.mkdir(exist_ok=True)
    code = meta.get("code", "set").lower()
    ruta = Path(ruta or REPORTES / f"limited_{code}{'_pool' if pool else ''}.html")
    ruta.write_text(doc, encoding="utf-8")
    print(f"Reporte guardado: {ruta} ({ruta.stat().st_size / 1e3:.0f} KB)")
    return ruta


def mostrar_reporte(ruta: Path, alto: int = 900) -> None:
    """Muestra el HTML dentro del notebook (iframe aislado para que el CSS no choque con DataLab)."""
    import warnings
    doc = Path(ruta).read_text(encoding="utf-8")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        display(HTML(f'<iframe srcdoc="{esc(doc, quote=True)}" style="width:100%;height:{alto}px;border:0;border-radius:12px"></iframe>'))